# External Drivers EDA + Model
 Analyze external factors' (Pricing, Policy, Regulation) effect on emission intensity.

---

💨 Emissions - Scope 1, Scope 2 (Location , Market)

🏠 Company Info - Country, Founding Year, Technology

💵 Price - Carbon, Electricity, Gas, Coal, Iron ore

🏭 ETS - EU Emissions Trading System - Regulates Permitted Green House Gas Emissions

📑 Policy - Points in time certain policies were 'enacted'

---

Key challenge: Limited temporal variation + EU-wide simultaneous shocks

---

FINDINGS:
1. POLICY FAILURE (DiD results):
   - ETS, CBAM, Green Deal show NO significant intensity reduction
   - Best treatment year: 2019 (p=0.0375) but effect disappears with controls
   - Conclusion: Regulatory measures haven't worked yet

2. MARKET SUCCESS (Granger results):
   - Coal price Granger-causes intensity (lag2 p=0.0015) → price↑ → intensity↓
   - Carbon price borderline (lag2 p=0.0018)
   - Conclusion: Economic incentives drive efficiency

3. TECHNOLOGY MATTERS (FE by tech):
   - BF-BOF: Age dominant (coef=0.037***), scale hurts efficiency
   - EAF: Price-sensitive, policy-responsive (IED, CSR significant)
   - Conclusion: Different decarbonization pathways needed

---

> Results should be interpreted cautiously given data limitations and model specifications.

---

## Setup & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import statsmodels.api as sm
from datetime import datetime

from statsmodels.formula.api import ols
from IPython.display import display
from linearmodels.panel import compare
from statsmodels.stats.outliers_influence import variance_inflation_factor

from functions import *

import warnings
warnings.filterwarnings('ignore')

In [ ]:
DATA_DIR = "../data/EDA/"

emissions = pd.read_excel(f"{DATA_DIR}emissions_steel_production.xls", sheet_name= 'Master_Dataset')
company_info = pd.read_excel(f"{DATA_DIR}emissions_steel_production.xls", sheet_name= 'Company_Profiles')
external = pd.read_excel(f"{DATA_DIR}external_drivers.xlsx", sheet_name='Sheet1')

# EDA

## Data cleaning

In [ ]:
fix_column_names(emissions)
fix_column_names(company_info)
fix_column_names(external)

In [ ]:
emissions['intensity1'] = emissions['scope1'] / emissions['production']
columns_emission = ['company', 'country', 'technology', 'year',
       'production', 'scope1', 'scope2_location', 'scope2_market',
       'scope_1_2_location', 'intensity1', 'intensity_location_co2e',
       'intensity_market_co2e']

company_info['age'] = datetime.now().year - company_info['founding_year']
df_company = pd.merge(emissions[columns_emission],company_info[['company','country','founding_year', 'age']],on='company', how='left',suffixes=['','_y'])
df_company.drop(df_company.filter(regex='_y$').columns, axis=1, inplace=True)
df_company.head()

In [ ]:
categorical_external_columns = ['cbam', 'ied_update', 'green_deal', 'nzia', 'csp_funding', 'gpp', 'csr',
       'paris', 'ksg', 'esrs', 'glasgow', 'govt_change_de', 'govt_change_it',
       'govt_change_se', 'govt_change_nl', 'govt_change_es', 'govt_change_fi',
       'govt_change_fr', 'govt_change_lu', 'govt_change_gb', 'govt_change_at',
       'eu_parl', 'eu_comm','govt_change']
external['green_deal'] = external['green_deal'] .replace({'communicated':1})

## Merge and save dataframe to csv

In [ ]:
df_og = pd.merge(df_company,external, on=['country', 'year'], how='left')
df_og[categorical_external_columns] = df_og[categorical_external_columns].fillna(0)
df_og.technology = df_og.technology.str.replace(' ','-')
df_og = df_og[df_og['intensity1'].notna()]

#save
# df_og.to_csv(f'{DATA_DIR}external_drivers_dataframe.csv')
df_og.head()

In [ ]:
df_og.columns

In [ ]:
overview_data(df_og)

#### Handle categorical
external_dummies : technology and country

df_enc : only country

In [ ]:
external_dummies = pd.get_dummies(external, columns=categorical_external_columns)
df_dummies = pd.get_dummies(df_og, columns=['technology', 'country'])
fix_column_names(df_dummies)
df_enc = pd.get_dummies(df_og, columns=['country'])
fix_column_names(df_enc)

### Special Case: Drop Companies
use only specific companies, original dataframe may contain more.

In [ ]:
df_og['company'].unique()

In [ ]:
df_og = df_og[df_og['company'].isin(['ArcelorMittal', 'Celsa Group',
       'Outokumpu', 'SSAB', 'Salzgitter AG', 'Tata Steel Nederland',
       'Voestalpine', 'Tata Steel UK', 'Acerinox EU',
       'Acciaierie d’Italia Holding', 'SIDENOR Group',
       'Feralpi Group', 'SHS Group'])]

# handle available scope data

In [ ]:
df_og.technology.unique()

In [ ]:
df_og.groupby('company').count()

In [ ]:
# # Number of available scope1 datapoints for each company
# for comp in df_og['company'].unique():
#     x = df_og[df_og['company'] == comp]['scope1'].notna().sum()
#     print(comp, x)

In [ ]:
# # Set Threshold
# df_filtered_scope1 = df_og.groupby('company').filter(lambda x: x['scope1'].notna().sum() >= 3)
# # for now on minimum -> 3
# df_scope1 = df_filtered_scope1.dropna(subset=['scope1']).copy()

# df_scope1['intensity1'] = df_scope1['scope1'] / df_scope1['production']
# df_scope1.groupby('company').count()

## Visualization 
 
Company emissions over time

In [ ]:
df_og.head(2)

In [ ]:
plot_panel_timeseries(df_og, value_col='intensity1')

In [ ]:
facet_panel_plots(df_og, value_col='intensity1')

In [ ]:
prices = ['electricity_price','carbon_price','coal_price_australia', 'iron_ore_price', 'natural_gas_price']

# Aggregate by year
df_agg = df_og.groupby('year')[prices].mean().reset_index()

### Price development

In [ ]:
# Prices Over Time
plt.figure(figsize=(12, 6))
plt.plot(df_agg['year'], df_agg['carbon_price'], label='Carbon Price', linewidth=2, color='green')
plt.plot(df_agg['year'], df_agg['iron_ore_price'], label='Iron Ore Price', linewidth=2, color='brown')
plt.plot(df_agg['year'], df_agg['electricity_price'], label='Electricity Price', linewidth=2, color='blue')
plt.plot(df_agg['year'], df_agg['coal_price_australia'], label='Coal Price (Australia)', linewidth=2, color='black')
plt.plot(df_agg['year'], df_agg['natural_gas_price'], label='Natural Gas Price', linewidth=2, color='orange')
plt.xlabel('Year')
plt.ylabel('Price (€ per unit)')
plt.title('Prices Over Time')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Divide Prices into 2 groups due to scale difference

In [ ]:
# First graph: Carbon price and Iron ore price
plt.figure(figsize=(12, 6))
plt.plot(df_agg['year'], df_agg['carbon_price'], label='Carbon Price', linewidth=2, color='green')
plt.plot(df_agg['year'], df_agg['iron_ore_price'], label='Iron Ore Price', linewidth=2, color='brown')
plt.xlabel('Year')
plt.ylabel('Price (€ per unit)')
plt.title('Carbon Price vs Iron Ore Price Over Time')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Second graph: The other three prices
plt.figure(figsize=(12, 6))
plt.plot(df_agg['year'], df_agg['electricity_price'], label='Electricity Price', linewidth=2, color='blue')
plt.plot(df_agg['year'], df_agg['coal_price_australia'], label='Coal Price (Australia)', linewidth=2, color='black')
plt.plot(df_agg['year'], df_agg['natural_gas_price'], label='Natural Gas Price', linewidth=2, color='orange')
plt.xlabel('Year')
plt.ylabel('Price (€ per unit)')
plt.title('Energy Prices Over Time')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### ETS

In [ ]:
ets = ['ets_coke_production', 'ets_metal_ore',
       'ets_iron_steel', 'ets_hydrogen_gas', 'ets']

# Aggregate by year - ETS
df_agg_e = df_og.groupby('year')[ets].mean().reset_index()

In [ ]:
plt.figure(figsize=(12, 8))
for e in ets:
    plt.plot(df_agg_e['year'], df_agg_e[e], label=e)

plt.xlabel('Year')
plt.ylabel('ETS')
plt.title('ETS Over Time')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Divide Technologies

Since we saw a grouping of intensity by technology used, we will treat them as individual groups here.

In [ ]:
plot_panel_timeseries(df_og, color_by='technology', value_col='intensity1')

In [ ]:
tech_summary = (
    df_og
    .groupby('technology')['intensity1']
    .agg(['mean','median','min','max'])
    .round(3)
)

tech_summary

In [ ]:
df_eaf = df_og[df_og['technology'].str.contains('EAF', case=False)]
df_bf = df_og[df_og['technology'].str.contains('BF-BOF', case=False)]

In [ ]:

fig, (ax1, ax2) = plt.subplots(2,1, figsize=(12,8))
for comp in df_eaf['company'].unique():
    comp_data = df_eaf[df_eaf['company']== comp].sort_values('year')
    ax1.plot(comp_data['year'],comp_data['intensity1'], label=comp)
for comp in df_bf['company'].unique():
    comp_data = df_bf[df_bf['company']== comp].sort_values('year')
    ax2.plot(comp_data['year'], comp_data['intensity1'], label = comp)


# Formatting
ax1.set_title('EAF Companies - Intensity1')
ax2.set_title('BF-BOF Companies - Intensity1')
ax1.set_ylabel('Intensity')
ax2.set_ylabel('Intensity')
ax2.set_xlabel('Year')

# Add legends (consider putting them outside if many companies)
ax1.legend(fontsize=8, bbox_to_anchor=(1.0, 1.0))
ax2.legend(fontsize=8, bbox_to_anchor=(1.0, 1.0))

plt.tight_layout()


Which group has more fluctuations

In [ ]:
def calculate_stability_metrics(df, company_col='company', intensity_col='intensity1'):
    metrics = {}
    
    for company in df[company_col].unique():
        company_data = df[df[company_col] == company].sort_values('year')
        intensity = company_data[intensity_col]
        
        if len(intensity) > 1:
            metrics[company] = {
                'volatility': intensity.std(),  # Standard deviation
                'cv': intensity.std() / intensity.mean(),  # Coefficient of variation
                'max_change': intensity.diff().abs().max(),  # Max year-to-year change
                'trend_slope': np.polyfit(range(len(intensity)), intensity, 1)[0],  # Slope
                'range': intensity.max() - intensity.min(),  # Total range
                'years': len(intensity)  # Number of observations
            }
    
    return pd.DataFrame(metrics).T

# Calculate for both groups
eaf_stability = calculate_stability_metrics(df_eaf)
bf_stability = calculate_stability_metrics(df_bf)

print("EAF Stability Summary:")
print(eaf_stability.describe())
print("\nBF Stability Summary:")
print(bf_stability.describe())

In [ ]:
print("Which group is more stable?")
print(f"Average volatility - EAF: {eaf_stability['volatility'].mean():.3f}, BF: {bf_stability['volatility'].mean():.3f}")
print(f"Average CV - EAF: {eaf_stability['cv'].mean():.3f}, BF: {bf_stability['cv'].mean():.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))  # Adjusted height to 5 for better proportions

# Plot 1: Volatility
axes[0].boxplot([eaf_stability['volatility'].dropna(), bf_stability['volatility'].dropna()])
axes[0].set_xticklabels(['EAF', 'BF-BOF'])
axes[0].set_title('Volatility (Std Dev)')

# Plot 2: Coefficient of Variation
axes[1].boxplot([eaf_stability['cv'].dropna(), bf_stability['cv'].dropna()])
axes[1].set_xticklabels(['EAF', 'BF-BOF'])
axes[1].set_title('Coefficient of Variation')

plt.tight_layout()
plt.show()

# Modeling - PanelOLS

- fixed effects
- two way fixed effects
- random effects

For our purpose a fixed effects model is appropriate, but all three were tested to see their effects.

During the conception of the model high collinearity was found and further investigated. Due to this the ets categories were removed as features.


In [ ]:
# Check for multicollinearity before FE model
def check_vif(df, features):
    """Calculate Variance Inflation Factor for feature set"""
    X = df[features].fillna(0)
    vif_data = pd.DataFrame()
    vif_data['feature'] = features
    vif_data['VIF'] = [variance_inflation_factor(X.values, i) 
                       for i in range(len(features))]
    vif_data['status'] = vif_data['VIF'].apply(
        lambda x: 'HIGH (≥10)' if x >= 10 else 'OK')
    return vif_data.sort_values('VIF', ascending=False)

# Test your variables
features = ['ets','ets_coke_production', 'ets_metal_ore', 'ets_iron_steel',
       'ets_hydrogen_gas', 'hydrogen_use', 'free_allocation',
       'ets_free_ratio']
vif_results = check_vif(df_og, features)
print("VIF Check for Policy Variables:")
print(vif_results)

# Also check correlation matrix
corr_matrix = df_og[features].corr().round(2)
print("\nCorrelation Matrix (abs > 0.8 highlighted):")
print(corr_matrix.style.applymap(lambda x: 'background-color: yellow' 
                                 if abs(x) > 0.8 and x != 1 else ''))

Can we include production as a variable?

In [ ]:
# Verify no mathematical identity
print(f"Correlation(intensity1, production): {df_dummies['intensity1'].corr(df_dummies['production']):.3f}")
print(f"Correlation(scope1, production): {df_dummies['scope1'].corr(df_dummies['production']):.3f}")

# Production vs intensity scatter
plt.scatter(df_dummies['production'], df_dummies['intensity1'], alpha=0.5)
plt.xlabel('Production')
plt.ylabel('Intensity')
plt.show()

## CRITICAL MODEL DIAGNOSTICS:

1. FE_TW MODEL FAILURE:
- R²(within) = -599 → severe misspecification
- Time FE likely absorbing all variation when combined with entity FE
- coal_price coefficient = 80.165 (implausibly large) → numerical instability

2. RE MODEL OVERFITTING:
- R²(between) = 1.000 → perfect between-entity fit (suspicious)
- RE absorbs time-invariant tech/country variables that FE cannot
- technology_* coefficients large but FE drops them (time-invariant)



In [ ]:
trsh=12

x_vars_clean = [
    'production','age', 'sci',
    'technology_bf_bof', 'technology_eaf', 'technology_eaf_stainless',
    'carbon_price', 'electricity_price','coal_price_australia', 'iron_ore_price', 'natural_gas_price',
    'ets', 'cbam', 'ied_update', 'green_deal', 'nzia','csp_funding', 'gpp', 'csr', 'paris', 'ksg', 'esrs', 'glasgow',
    'govt_change', 'eu_parl']

fe_result = EffectsModel(df_dummies,y='intensity1', x_vars= x_vars_clean, threshold_obs=trsh, check_rank=False)
re_result=EffectsModel(df_dummies,y='intensity1', x_vars= x_vars_clean, threshold_obs=trsh, model= 'random', check_rank=False)
fetw_result =EffectsModel(df_dummies,y='intensity1', x_vars= x_vars_clean, threshold_obs=trsh, model= 'twoway',check_rank=False)

# Compare
comparison = compare({'FE': fe_result, 'FE_TW' : fetw_result, 'RE': re_result}) 
print(comparison.summary)

In [ ]:
coeff = fe_result.params
ci = fe_result.conf_int()
coef_plot(coeff, ci)

# MAIN TAKEAWAY:

Fixed effects analysis reveals:
1. Within-firm changes: Only firm AGE significantly affects carbon intensity (Age is highly correlated with what technology is used)
2. Policy variables (ETS, CBAM, etc.) show no significant within-firm effects
3. Production scale has mild negative effect on intensity

This suggests decarbonization efforts haven't yet produced measurable
within-company intensity reductions beyond age-related effects.

GOING FORWARD:
- Focus on FE model (entity effects only)
- Drop FE_TW (two-way FE) - fails due to limited time periods
- RE results unreliable (overfits between variation)

## NEXT STEPS:

1. Test dynamic models with lagged intensity
2. Interact policies with technology type (BF-BOF vs EAF)
3. Event study around specific policy implementation years

In [ ]:
# Result
# x_vars to use

x_vars_clean = [
    'production','age', 'sci',
    'carbon_price', 'electricity_price','coal_price_australia', 'iron_ore_price', 'natural_gas_price',
    'ets', 'cbam', 'ied_update', 'green_deal', 'nzia','csp_funding', 'gpp', 'csr', 'paris', 'ksg', 'esrs', 'glasgow',
    'govt_change', 'eu_parl']

does the minimum amount of available observations matter?

In [ ]:
t3 = EffectsModel(df_dummies,y='intensity1', x_vars= x_vars_clean, threshold_obs=3, check_rank=False)
t6 = EffectsModel(df_dummies,y='intensity1', x_vars= x_vars_clean, threshold_obs=6, check_rank=False)
t9 = EffectsModel(df_dummies,y='intensity1', x_vars= x_vars_clean, threshold_obs=9,check_rank=False)
t12 = EffectsModel(df_dummies,y='intensity1', x_vars= x_vars_clean, threshold_obs=12,check_rank=False)

comparison = compare({'3': t3, '6' : t6, '9': t9, '12':t12}) 
comparison.summary

In [ ]:
# Result - Threshold
trsh=3

Quick check of the other scope data we have available

In [ ]:
intensity1 = EffectsModel(df_dummies,y='intensity1', x_vars= x_vars_clean, threshold_obs=trsh, check_rank=False)
intloc = EffectsModel(df_dummies,y='intensity_location_co2e', x_vars= x_vars_clean, threshold_obs=trsh, check_rank=False)
intmarket = EffectsModel(df_dummies,y='intensity_market_co2e', x_vars= x_vars_clean, threshold_obs=trsh,check_rank=False)

comp = compare({'intensity 1': intensity1, 'intensity scope 2 location' : intloc, 'intensity scope 2 market': intmarket})
comp.summary

In [ ]:
# Other variable combinations
# 
#  EU Price
x1c = [
    'age',
    'carbon_price', 'electricity_price_eu','coal_price_australia', 
    'iron_ore_price', 'natural_gas_price_eu',
    'ets', 'cbam', 'ied_update', 'green_deal', 'nzia','csp_funding', 'gpp', 'csr', 'paris', 'ksg', 'esrs', 'glasgow',
    'govt_change', 'eu_parl', 
    'technology_bf_bof', 'technology_eaf', 'technology_eaf_stainless']
# prices country. removed: hydrogen_use[nans], sci makes the r2 worse but still fine
x2c = [
    'age',
    'carbon_price', 'electricity_price','coal_price_australia', 'iron_ore_price', 'natural_gas_price',
    'ets', 'cbam', 'ied_update', 'green_deal', 'nzia','csp_funding', 'gpp', 'csr', 'paris', 'ksg', 'esrs', 'glasgow',
    'govt_change', 'eu_parl', 
    'technology_bf_bof', 'technology_eaf', 'technology_eaf_stainless']

x3c = ['age', 
       'carbon_price', 'electricity_price','coal_price_australia', 'iron_ore_price', 'natural_gas_price',
       'crude_steel_production',
       'ets', 'ied_update',
       'govt_change', 'eu_parl'
       ]

x1 = EffectsModel(df_dummies,y='intensity1', x_vars= x1c, threshold_obs=trsh, check_rank=False)
x2 = EffectsModel(df_dummies,y='intensity1', x_vars= x2c, threshold_obs=trsh, check_rank=False)
x3 = EffectsModel(df_dummies,y='intensity1', x_vars= x3c, threshold_obs=trsh, check_rank=False)
# Compare
comparison = compare({'EU Price': x1, 'Local Price' : x2, 'removed events':x3})
print(comparison.summary)

# Divide by technology
#### KEY TECHNOLOGY DIFFERENCES REVEALED:

1. BF-BOF FIRMS (Blast Furnace):
   - Strong AGE effect (coef=0.037, t=25***) → older BF plants much higher intensity
   - PRODUCTION increases intensity (coef=0.0008, t=3.0**) → scale hurts efficiency
   - ETS reduces intensity (coef=-0.051, t=-1.8*) → carbon pricing works for BF
   - CBAM reduces intensity (coef=-0.034, t=-1.7) → border adjustment effective

2. EAF FIRMS (Electric Arc):
   - Weak AGE effect (coef=0.004, t=4.0***) → age matters less
   - CARBON PRICE reduces intensity (coef=-0.0014, t=-2.0*) → price sensitivity
   - IED UPDATE increases intensity (coef=0.048, t=2.5**) → regulation burden?
   - CSR reduces intensity (coef=-0.035, t=-2.1*) → reporting helps EAF
   - PARIS increases intensity (coef=0.053, t=2.1**) → counterintuitive

3. TECHNOLOGY COMPARISON:
   - BF-BOF: Driven by AGE, ETS, scale inefficiencies
   - EAF: Driven by CARBON PRICE, POLICIES (IED, CSR, Paris)
   - Policies affect technologies DIFFERENTLY (important for targeting)

4. MODEL FIT:
   - BF-BOF R²=0.565 → model explains 56% of within-firm variation
   - EAF R²=0.501 → explains 50% of within-firm variation
   - Both better than pooled model (R²=0.379) → technology stratification helps

POLICY IMPLICATIONS:
   - Carbon pricing: Effective for EAF, borderline for BF-BOF
   - Border adjustment (CBAM): Works for BF-BOF only
   - Reporting (CSR): Helps EAF efficiency
   - Age matters: BF-BOF plants need modernization

In [ ]:
bf_result = EffectsModel(df_bf,y='intensity1', x_vars= x_vars_clean, threshold_obs=trsh, check_rank=False)
eaf_result=EffectsModel(df_eaf,y='intensity1', x_vars= x_vars_clean, threshold_obs=trsh, check_rank=False)
# Compare
comparison = compare({'bf-bof': bf_result, 'eaf' : eaf_result, 'all': fe_result}) 
print(comparison.summary)

In [ ]:
coeff_bf = bf_result.params
ci_bf = bf_result.conf_int()
coef_plot(coeff_bf, ci_bf, title='BF-BOF')

coeff_eaf = eaf_result.params
ci_eaf = eaf_result.conf_int()
coef_plot(coeff_eaf, ci_eaf, title='EAF')

# Policies - DiD

- 2014 - IED Update + GPP criteria

- 2016 - Paris Agreement

- 2019 - German KSG

- 2020 - EU Green Deal

- 2021 - Glasgow Pact + CSP Funding

- 2023 - CBAM + NZIA + ESRS

Methodological concerns regarding Difference in Difference, due to assumption that companies which use EAF are not affected by shocks.

Treatment group : BF-BOF

Control Group : EAF

Since Data is available from 2013 - 2024

In [ ]:
# since code created in different notebook, naming convention is different
df = df_og.copy()

In [ ]:
df['treated'] = (df['technology'].str.contains('BF-BOF')).astype(int)

# Define range of possible treatment years
treatment_years = range(2015, 2023) 
results = {}

for year in treatment_years:
    df['post'] = (df['year'] >= year).astype(int)
    df['treated_post'] = df['treated'] * df['post']
    # Basic DiD model
    model1 = ols('intensity1 ~ treated + post + treated_post', data=df).fit()
    # Model with controls
    model2 = ols('intensity1 ~ treated + post + treated_post + carbon_price + electricity_price + C(company)', data=df).fit()
    # Store key results
    results[year] = {
        'basic_coef': model1.params['treated_post'],
        'basic_p': model1.pvalues['treated_post'],
        'basic_rsquared': model1.rsquared,
        'controls_coef': model2.params['treated_post'],
        'controls_p': model2.pvalues['treated_post'],
        'controls_rsquared': model2.rsquared,
        'n_obs': len(model1.model.endog)
    }

# Convert to DataFrame for easier analysis
results_df = pd.DataFrame(results).T
print("Results for all treatment years:")
display(results_df.round(4))

In [ ]:
# Criteria 1: Most statistically significant negative effect
results_df['sig_score'] = -np.log(results_df['controls_p']) * results_df['controls_coef'].abs()
best_by_sig = results_df['sig_score'].idxmax()

# Criteria 2: Most negative coefficient with p < 0.1
sig_results = results_df[results_df['controls_p'] < 0.1]
best_by_coef = sig_results['controls_coef'].idxmin() if not sig_results.empty else None

# Criteria 3: Highest R² in controlled model
best_by_rsquared = results_df['controls_rsquared'].idxmax()

print(f"\nBest year by significance score: {best_by_sig}")
print(f"Best year by negative coefficient (p<0.1): {best_by_coef}")
print(f"Best year by R-squared: {best_by_rsquared}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Plot 1: Treatment effect coefficient
axes[0, 0].plot(results_df.index, results_df['controls_coef'], 'o-', markerfacecolor='white')
axes[0, 0].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
axes[0, 0].set_xlabel('Treatment Year')
axes[0, 0].set_ylabel('DiD Coefficient')
axes[0, 0].set_title('Treatment Effect by Year (with controls)')

# Highlight significant years
sig_years = results_df[results_df['controls_p'] < 0.1].index
for year in sig_years:
    axes[0, 0].plot(year, results_df.loc[year, 'controls_coef'], 'o', color='red', markersize=10)

# Plot 2: P-values
axes[0, 1].semilogy(results_df.index, results_df['controls_p'], 's-', markerfacecolor='white')
axes[0, 1].axhline(y=0.05, color='red', linestyle='--', alpha=0.7, label='p=0.05')
axes[0, 1].axhline(y=0.10, color='orange', linestyle='--', alpha=0.7, label='p=0.10')
axes[0, 1].set_xlabel('Treatment Year')
axes[0, 1].set_ylabel('P-value (log scale)')
axes[0, 1].set_title('Statistical Significance by Year')
axes[0, 1].legend()

# Plot 3: R-squared comparison
axes[1, 0].plot(results_df.index, results_df['basic_rsquared'], 'o-', label='Basic model')
axes[1, 0].plot(results_df.index, results_df['controls_rsquared'], 's-', label='With controls')
axes[1, 0].set_xlabel('Treatment Year')
axes[1, 0].set_ylabel('R-squared')
axes[1, 0].set_title('Model Fit by Year')
axes[1, 0].legend()

# Plot 4: Observations per year
axes[1, 1].bar(results_df.index, results_df['n_obs'])
axes[1, 1].set_xlabel('Treatment Year')
axes[1, 1].set_ylabel('Number of Observations')
axes[1, 1].set_title('Sample Size by Treatment Year')

plt.tight_layout()
plt.show()

# Granger Causality
$$Y_t = α + β₁Y_{t-1} + β₂Y_{t-2} + γ₁X_{t-1} + γ₂X_{t-2} + ε$$
$R^2$ within will tell how much of the within company variation can be explained with 


### BF-BOF Technology

In [ ]:
carbon = lag_model(df_bf,y='intensity1',x='carbon_price')
electricity = lag_model(df_bf,'intensity1','electricity_price')
coal = lag_model(df_bf,'intensity1','coal_price_australia')
iron_ore = lag_model(df_bf,'intensity1','iron_ore_price')
gas = lag_model(df_bf,'intensity1','natural_gas_price')

In [ ]:
# Print Granger causality summary
print("=" * 80)
print("GRANGER CAUSALITY TEST RESULTS")
print("=" * 80)
print(f"{'Variable':<20} {'F-pval':<10} {'Lag1 t':<10} {'Lag1 p':<10} {'Lag2 t':<10} {'Lag2 p':<10} {'Result'}")
print("-" * 80)

for name, model_dict in [('Carbon Price', carbon), 
                          ('Electricity Price', electricity),
                          ('Coal Price', coal),
                          ('Iron Ore Price', iron_ore),
                          ('Natural Gas Price', gas)]:
    f_p = f"{model_dict['f_pvalue']:.4f}" if model_dict['f_pvalue'] is not None else "N/A"
    print(f"{name:<20} {f_p:<10} {model_dict['x_lag1_tstat']:>9.2f} {model_dict['x_lag1_pval']:>9.4f} "
          f"{model_dict['x_lag2_tstat']:>9.2f} {model_dict['x_lag2_pval']:>9.4f}  {model_dict['granger']}")

print("=" * 80)
print("\nInterpretation:")
print("✅ SIGNIFICANT     = Variable Granger-causes emission intensity (joint F-test p < 0.05)")
print("❌ NOT SIGNIFICANT = No evidence of Granger causality (joint F-test p ≥ 0.05)")
print("⚠️ CHECK MANUALLY  = F-test failed, but at least one lag is individually significant")

# show comparison table
comparison = compare({'carbon': carbon['result'], 
                      'electricity': electricity['result'], 
                      'coal': coal['result'], 
                      'iron ore': iron_ore['result'], 
                      'gas': gas['result']})
print("\n" + "=" * 80)
print("DETAILED REGRESSION COMPARISON")
print("=" * 80)
print(comparison.summary)

### EAF Technology

In [ ]:
electricity = lag_model(df_eaf,'intensity1','electricity_price')
coal = lag_model(df_eaf,'intensity1','coal_price_australia')
iron_ore = lag_model(df_eaf,'intensity1','iron_ore_price')
gas = lag_model(df_eaf,'intensity1','natural_gas_price')

# Print Granger causality summary
print("=" * 80)
print("GRANGER CAUSALITY TEST RESULTS")
print("=" * 80)
print(f"{'Variable':<20} {'F-pval':<10} {'Lag1 t':<10} {'Lag1 p':<10} {'Lag2 t':<10} {'Lag2 p':<10} {'Result'}")
print("-" * 80)

for name, model_dict in [('Carbon Price', carbon), 
                          ('Electricity Price', electricity),
                          ('Coal Price', coal),
                          ('Iron Ore Price', iron_ore),
                          ('Natural Gas Price', gas)]:
    f_p = f"{model_dict['f_pvalue']:.4f}" if model_dict['f_pvalue'] is not None else "N/A"
    print(f"{name:<20} {f_p:<10} {model_dict['x_lag1_tstat']:>9.2f} {model_dict['x_lag1_pval']:>9.4f} "
          f"{model_dict['x_lag2_tstat']:>9.2f} {model_dict['x_lag2_pval']:>9.4f}  {model_dict['granger']}")

print("=" * 80)
print("\nInterpretation:")
print("✅ SIGNIFICANT     = Variable Granger-causes emission intensity (joint F-test p < 0.05)")
print("❌ NOT SIGNIFICANT = No evidence of Granger causality (joint F-test p ≥ 0.05)")
print("⚠️ CHECK MANUALLY  = F-test failed, but at least one lag is individually significant")

# show comparison table
comparison = compare({'carbon': carbon['result'], 
                      'electricity': electricity['result'], 
                      'coal': coal['result'], 
                      'iron ore': iron_ore['result'], 
                      'gas': gas['result']})
print("\n" + "=" * 80)
print("DETAILED REGRESSION COMPARISON")
print("=" * 80)
print(comparison.summary)

In [ ]:
# SUMMARY TABLE
summary = pd.DataFrame({
    'Method': ['Difference-in-Differences', 'Granger Causality'],
    'Question': ['Do policies reduce intensity?', 'Do prices predict intensity?'],
    'Key Finding': ['Policies have minimal/no effect', 'Coal/carbon prices matter (lagged)'],
    'Strongest Signal': ['2019 treatment (p=0.0375)', 'Coal price lag2 (p=0.0015)'],
    'Interpretation': [
        'Policy interventions ineffective so far',
        'Market forces drive efficiency with 2-year lag'
    ]
})

summary


COMPREHENSIVE STEEL DECARBONIZATION FINDINGS:

1. POLICY FAILURE (DiD results):
   - ETS, CBAM, Green Deal show NO significant intensity reduction
   - Best treatment year: 2019 (p=0.0375) but effect disappears with controls
   - Conclusion: Regulatory measures haven't worked yet

2. MARKET SUCCESS (Granger results):
   - Coal price Granger-causes intensity (lag2 p=0.0015) → price↑ → intensity↓
   - Carbon price borderline (lag2 p=0.0018)
   - Conclusion: Economic incentives drive efficiency

3. TECHNOLOGY MATTERS (FE by tech):
   - BF-BOF: Age dominant (coef=0.037***), scale hurts efficiency
   - EAF: Price-sensitive, policy-responsive (IED, CSR significant)
   - Conclusion: Different decarbonization pathways needed

4. DATA ISSUES:
   - High VIF in ETS variables → keep only aggregate 'ets'
   - scope1/production collinearity → model intensity directly
   - Negative R²(between) in FE → within-firm variation only


